In [1]:
import re
import pandas as pd
from collections import defaultdict
import os

# Function to extract data from the structure info file and group by space group
def extract_structure_data(file_path, functional_name, grouped_data, energy_data):
    with open(file_path, 'r') as file:
        data = file.read()

    # Split data by separators
    phases = data.split('-----------------------------------------')

    for phase in phases:
        # Extract space group, lattice constants, angles, and energy (E0)
        space_group_match = re.search(r'International:\s*(\S+)', phase)
        lattice_constants_match = re.search(r'Lattice Constants:\s*([\d\.]+\s+[\d\.]+\s+[\d\.]+)', phase)
        lattice_angles_match = re.search(r'Lattice Angles:\s*([\d\.]+\s+[\d\.]+\s+[\d\.]+)', phase)
        energy_match = re.search(r'E0=\s*([-\d\.E+]+)', phase)

        if space_group_match and lattice_constants_match and lattice_angles_match and energy_match:
            space_group = space_group_match.group(1)
            lattice_constants = list(map(float, lattice_constants_match.group(1).split()))
            lattice_angles = list(map(float, lattice_angles_match.group(1).split()))
            energy = float(energy_match.group(1))

            if space_group not in grouped_data:
                grouped_data[space_group] = defaultdict(lambda: [float('nan')] * 6)

            grouped_data[space_group][functional_name] = lattice_constants + lattice_angles

            if space_group not in energy_data:
                energy_data[space_group] = defaultdict(lambda: float('nan'))

            energy_data[space_group][functional_name] = energy

# Function to save the grouped data into a CSV file for lattice constants and angles
def save_to_csv(file_paths, output_structure_csv, output_energy_csv="different_partial_doping_energy_data.csv"):
    grouped_data = defaultdict(lambda: defaultdict(list))
    energy_data = defaultdict(lambda: defaultdict(float))

    # Extract data from each file
    for file_path, functional_name in file_paths:
        extract_structure_data(file_path, functional_name, grouped_data, energy_data)

    ### ----- STRUCTURE DATA -----
    # Load existing structure data if file exists
    if os.path.exists(output_structure_csv):
        existing_df = pd.read_csv(output_structure_csv)
        existing_space_groups = existing_df["Space Group"].tolist()
        existing_columns = existing_df.columns.tolist()
    else:
        existing_df = pd.DataFrame()
        existing_space_groups = []
        existing_columns = ["Space Group"]

    # Get all unique functionals from file_paths
    all_functionals = list(set([func for _, func in file_paths]))

    # Create headers
    for func in all_functionals:
        for label in ['a', 'b', 'c', 'alpha', 'beta', 'gamma']:
            col = f"{func} {label}"
            if col not in existing_columns:
                existing_columns.append(col)

    all_data = []
    for space_group, func_data in grouped_data.items():
        row = {'Space Group': space_group}
        for func in all_functionals:
            values = func_data.get(func, [float('nan')] * 6)
            for val, label in zip(values, ['a', 'b', 'c', 'alpha', 'beta', 'gamma']):
                row[f"{func} {label}"] = val
        all_data.append(row)

    new_df = pd.DataFrame(all_data)

    # Merge with existing if needed
    if not existing_df.empty:
        merged_df = pd.merge(existing_df, new_df, on="Space Group", how="outer")
    else:
        merged_df = new_df

    # merged_df = merged_df[existing_columns + [col for col in new_df.columns if col not in existing_columns and col != 'Space Group']]
    # Only include columns that actually exist in merged_df
    valid_existing_columns = [col for col in existing_columns if col in merged_df.columns]
    new_cols = [col for col in new_df.columns if col not in valid_existing_columns and col != 'Space Group']
    merged_df = merged_df[valid_existing_columns + new_cols]

    merged_df.to_csv(output_structure_csv, index=False)

    ### ----- ENERGY DATA -----
    if os.path.exists(output_energy_csv):
        energy_df = pd.read_csv(output_energy_csv)
        existing_energies = energy_df.set_index("Space Group").to_dict("index")
    else:
        energy_df = pd.DataFrame()
        existing_energies = {}

    energy_rows = {}
    for space_group, func_energies in energy_data.items():
        row = {"Space Group": space_group}
        for func in all_functionals:
            row[f"{func} Energy"] = func_energies.get(func, float('nan'))
        energy_rows[space_group] = row

    # Combine with existing energy data
    for sg, row in existing_energies.items():
        if sg not in energy_rows:
            energy_rows[sg] = {"Space Group": sg}
        energy_rows[sg].update(row)

    energy_df = pd.DataFrame(list(energy_rows.values()))

    # Relative energies
    for func in all_functionals:
        col = f"{func} Energy"
        if col in energy_df.columns:
            min_energy = energy_df[col].min()
            min_sg = energy_df.loc[energy_df[col] == min_energy, "Space Group"].values[0]
            print(f"Minimum energy for {func}: {min_energy}, corresponding to space group: {min_sg}")
            energy_df[col] = energy_df[col] - min_energy

    energy_df = energy_df.sort_values(by=f"{all_functionals[0]} Energy")
    energy_df.to_csv(output_energy_csv, index=False)


file_paths = [
    ('h1z3o8__doping_structure_info.txt', 'H1Z3O8'),
    ('h3z1o8__doping_structure_info.txt', 'H3Z1O8'),
]

save_to_csv(file_paths, 'different_partial_doping_structure_data_grouped_.csv')

Minimum energy for H1Z3O8: -119.6378, corresponding to space group: P2_1/c
Minimum energy for H3Z1O8: -128.64179, corresponding to space group: P2_1/c
